# Train two 0.8B students on Kaggle T4s

Two students from the same states and oracle labels, one per GPU, in parallel: `sft_features` (nearest ghost per option) and `sft_ghosts` (every ghost).
Then each adapter is merged and converted to a Q8_0 GGUF for Ollama.

Settings: Accelerator = GPU T4 x2, Internet = on. Run it in the background with **Save Version > Save & Run All**, then download `/kaggle/working/out.zip` from the version's Output tab.

Measured on a T4 (fp32, since Unsloth refuses fp16 for Qwen3.5): the first step takes about 3.5 minutes to compile kernels, then about 21 s per step with `--grad-checkpoint true`. `false` runs out of memory even with 8-row micro-batches. 600 steps is where validation agreement stopped rising in the MLX guide.

In [ ]:
# Settings
DATASET = 'liudongtony/pacman-sft-p1'   # the Kaggle dataset uploaded from data/sft_p1.zip
STUDENTS = ['sft_features', 'sft_ghosts']
STEPS = 600
TRAIN_ARGS = ['--steps', str(STEPS), '--grad-checkpoint', 'true', '--eval-every', '200', '--val-limit', '1000', '--log-every', '10']
RUN_TRIAL = False                       # True: a 6-step check before the real run
LLAMA_CPP_TAG = 'b11298'                 # the llama.cpp tag the iOS app pins

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print(torch.__version__, torch.cuda.get_device_name(0), 'capability', torch.cuda.get_device_capability(0))

In [ ]:
# The versions in training/pyproject.toml.
!pip install -q "unsloth>=2026.9" "transformers>=5.5" "peft>=0.17"

In [ ]:
import os, subprocess
REPO = '/kaggle/working/decision-pacman'
if os.path.isdir(REPO):
    subprocess.run(['git', '-C', REPO, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '-b', 'local/learning-poc', 'https://github.com/dongliutony/decision-pacman.git', REPO], check=True)
os.chdir(REPO)
print(subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True).stdout)

In [ ]:
import glob
# Look where Kaggle mounts inputs first. A newly added input can show up as an empty folder,
# so fall back to downloading the dataset (needs Internet on).
found = sorted(glob.glob('/kaggle/input/**/sft_*/train.jsonl', recursive=True))
if not found:
    import kagglehub
    path = kagglehub.dataset_download(DATASET)
    found = sorted(glob.glob(os.path.join(path, '**/sft_*/train.jsonl'), recursive=True))
DATA = {os.path.basename(os.path.dirname(p)): os.path.dirname(p) for p in found}
assert set(STUDENTS) <= DATA.keys(), DATA
DATA

In [ ]:
import time

def train(students, extra, out_root):
    """Train one student per GPU at the same time and print progress every 10 minutes."""
    procs = {}
    for gpu, name in enumerate(students):
        os.makedirs(out_root, exist_ok=True)
        log = open(f'{out_root}/{name}.log', 'w')
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu)}
        cmd = ['python', 'training/train.py', '--data', DATA[name], '--out', f'{out_root}/{name}', *extra]
        procs[name] = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)
    t0 = last = time.time()
    while any(p.poll() is None for p in procs.values()):
        time.sleep(30)
        if time.time() - last < 600:
            continue
        last = time.time()
        for name in students:
            lines = [l.strip() for l in open(f'{out_root}/{name}.log') if l.startswith(('step ', 'eval step', 'before training'))]
            print(f'[{(time.time() - t0) / 60:.0f} min] {name}: {lines[-1] if lines else "loading"}', flush=True)
    for name, p in procs.items():
        print(f'== {name}: exit {p.returncode}')
        print(''.join([l for l in open(f'{out_root}/{name}.log') if l.startswith(('before training', 'eval step', 'saved', 'Traceback', 'torch.OutOfMemory'))]))
    return {name: p.returncode for name, p in procs.items()}

In [ ]:
if RUN_TRIAL:
    train(STUDENTS, ['--train-limit', '192', '--epochs', '1', '--grad-checkpoint', 'true', '--val-limit', '32', '--eval-every', '1000', '--log-every', '1'], '/kaggle/working/trial')

## The real run: about 4 hours for both students in parallel

In [ ]:
codes = train(STUDENTS, TRAIN_ARGS, '/kaggle/working/runs')
assert all(c == 0 for c in codes.values()), codes

## Merge each adapter and convert it to a Q8_0 GGUF

In [ ]:
LLAMA = '/kaggle/working/llama.cpp'
if not os.path.isdir(LLAMA):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', LLAMA_CPP_TAG, 'https://github.com/ggml-org/llama.cpp.git', LLAMA], check=True)
# The converter's own Python package; transformers stays the one installed above (Qwen3.5 needs 5.5 or later).
subprocess.run(['pip', 'install', '-q', '-e', f'{LLAMA}/gguf-py', 'sentencepiece'], check=True)
os.makedirs('/kaggle/working/gguf', exist_ok=True)
for name in STUDENTS:
    run = f'/kaggle/working/runs/{name}'
    subprocess.run(['python', 'training/merge.py', '--adapter', f'{run}/adapter', '--out', f'{run}/merged'], check=True)
    out = f'/kaggle/working/gguf/pacman-0.8b-{name.replace("sft_", "")}-q8_0.gguf'
    subprocess.run(['python', f'{LLAMA}/convert_hf_to_gguf.py', f'{run}/merged', '--outtype', 'q8_0', '--outfile', out], check=True)
!ls -la /kaggle/working/gguf

In [ ]:
# One file to download: both GGUFs, both adapters, and the training logs.
!cd /kaggle/working && zip -qr out.zip gguf runs/*.log runs/*/log.jsonl runs/*/adapter && ls -la out.zip